# MLOps Starter Notebook

This notebook is a beginner-friendly workspace for the operational side of machine learning.

In simple words, MLOps is the part that helps us manage models after we build them. It covers things like:
- tracking experiments
- saving and versioning models
- checking whether new data looks different from old data
- serving a model through an API
- monitoring whether the system is healthy

You do not need to know all of these tools already. Each section below explains what the tool is for, why we use it, and shows a small example.

## Included tools

- MLflow and Model Registry
- Weights & Biases
- DVC
- Evidently AI
- WhyLogs and WhyLabs client
- FastAPI starter service
- TensorFlow Serving and TorchServe endpoints
- Prometheus metrics


## Environment Check

Run this once to make sure the important Python packages are installed in the notebook environment.

If this cell works, the rest of the notebook has a good chance of working too.


In [1]:
# Import the main MLOps libraries we will use throughout this notebook.
import mlflow
import pandas as pd
import wandb
import whylogs

# Evidently is used for data drift and monitoring reports.
from evidently import Report
from evidently.presets import DataDriftPreset

# Print versions so you can quickly confirm the notebook environment is ready.
print("MLflow:", mlflow.__version__)
print("W&B:", wandb.__version__)
print("WhyLogs:", whylogs.__version__)


MLflow: 2.17.2
W&B: 0.22.3
WhyLogs: 1.6.4


## MLflow Tracking And Model Registry

MLflow helps us keep a record of our machine learning work.

Think of it as a lab notebook for models. It stores:
- parameters we used
- metrics we measured
- model files we saved
- model versions we want to keep and reuse later

The example below shows how to create a small run, save metrics, and register a model version.


In [2]:
# Use a local SQLite-backed MLflow store so runs and registry metadata persist under notebooks/experiments.
mlflow.set_tracking_uri("sqlite:////opt/spark/notebooks/experiments/mlflow.db")

# Point the registry to the same backend so model versions are stored in one place.
mlflow.set_registry_uri("sqlite:////opt/spark/notebooks/experiments/mlflow.db")

# Create or reuse a named experiment for notebook demos.
mlflow.set_experiment("mlops-starter")

# Start a lightweight run and log a couple of demo values.
with mlflow.start_run(run_name="tracking-demo"):
    mlflow.log_param("framework", "sklearn")
    mlflow.log_param("use_case", "marketing-churn")
    mlflow.log_metric("demo_metric", 0.91)

# Print the active URIs so you know where MLflow is writing state.
print("Tracking URI:", mlflow.get_tracking_uri())
print("Registry URI:", mlflow.get_registry_uri())


2026/04/12 06:05:06 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



Tracking URI: sqlite:////opt/spark/notebooks/experiments/mlflow.db
Registry URI: sqlite:////opt/spark/notebooks/experiments/mlflow.db


In [3]:
# Build a tiny demo model so we can show the end-to-end registration pattern.
import mlflow.sklearn
from sklearn.linear_model import LogisticRegression

# Create a very small sample dataset just for registry demonstration.
X_demo = pd.DataFrame({"tenure": [1, 6, 12, 24, 36], "monthly_charges": [85, 79, 72, 61, 55]})
y_demo = [1, 1, 0, 0, 0]

# Train a simple baseline classifier.
model = LogisticRegression()
model.fit(X_demo, y_demo)

# Register the model under a reusable name so future runs can create new versions.
with mlflow.start_run(run_name="registered-model-demo"):
    mlflow.log_param("model_type", "logistic_regression")
    mlflow.log_metric("training_rows", len(X_demo))
    model_info = mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="model",
        registered_model_name="telco_churn_model",
    )

# Show the registered model location so you can inspect it later in MLflow.
print("Registered model URI:", model_info.model_uri)
print("Registered model name: telco_churn_model")


2026/04/12 06:05:07 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Registered model URI: runs:/6683a5001bbb4238a1b463cc44a4d20c/model
Registered model name: telco_churn_model


Registered model 'telco_churn_model' already exists. Creating a new version of this model...
Created version '6' of model 'telco_churn_model'.


## Weights & Biases

Weights & Biases, often called W&B, is another tool for tracking machine learning experiments.

It is useful when you want a polished web dashboard to compare runs, charts, metrics, and model settings.

The W&B dashboard is not part of this Docker stack. It is a hosted UI at [https://wandb.ai](https://wandb.ai).

Typical flow from this notebook:
1. Set `WANDB_API_KEY` from a hidden prompt using `getpass`, or export it before starting Jupyter.
2. Start a run with `wandb.init(project=..., entity=...)`.
3. Log metrics, parameters, and artifacts.
4. Open your run in the W&B web dashboard.

Project URL pattern:
- `https://wandb.ai/<entity>/<project>`

Run URL pattern:
- `https://wandb.ai/<entity>/<project>/runs/<run_id>`


In [4]:
# Import the W&B SDK for hosted experiment tracking.
import getpass
import os
import wandb

# Capture the API key without echoing it into notebook output.
# If you already exported WANDB_API_KEY before starting Jupyter, this block is skipped.
if not os.environ.get("WANDB_API_KEY"):
    os.environ["WANDB_API_KEY"] = getpass.getpass("Enter your W&B API key: ")

# Start a run and attach metadata that helps organize your dashboard.
run = wandb.init(
    project="telco-customer-churn",
    entity=None,  # Replace with your W&B team or username if needed.
    job_type="training",
    tags=["marketing", "churn", "mlops"],
    config={
        "owner": "marketing-ml",
        "environment": "local",
        "model": "xgboost",
        "threshold": 0.30,
    },
)

# Log a few representative churn metrics and a business KPI.
wandb.log({
    "roc_auc": 0.84,
    "pr_auc": 0.63,
    "profit": 1250,
})

# Print direct dashboard links so you can jump from the notebook into W&B.
print("Project page:", f"https://wandb.ai/{run.entity}/{run.project}")
print("Run page:", run.url)

# Close the run cleanly so metrics are flushed to the dashboard.
wandb.finish()


Enter your W&B API key:  ········


wandb: Currently logged in as: leninworld (leninworld-labs) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Project page: https://wandb.ai/leninworld-labs/telco-customer-churn
Run page: https://wandb.ai/leninworld-labs/telco-customer-churn/runs/z1h4yh7q


pr_auc,▁
profit,▁
roc_auc,▁
pr_auc,0.63
profit,1250
roc_auc,0.84


## Evidently AI Drift Report

Evidently helps us compare old data and new data.

This matters because a model may perform worse if the data it sees in the real world starts to look different from the data it was trained on.

In this example:
- `reference_data` means the older baseline data
- `current_data` means the newer data we want to check
- the report tells us whether the distributions have drifted


In [5]:
# Create a small reference dataset that represents the baseline production distribution.
import warnings

reference_data = pd.DataFrame(
    {
        "monthly_charges": [45, 52, 68, 74, 88],
        "tenure": [24, 18, 12, 6, 3],
    }
)

# Create a second dataset that simulates newer live data.
current_data = pd.DataFrame(
    {
        "monthly_charges": [55, 61, 70, 89, 95],
        "tenure": [20, 16, 8, 4, 2],
    }
)

# Suppress SciPy divide-by-zero warnings that can appear on tiny synthetic demo samples.
warnings.filterwarnings("ignore", message="divide by zero encountered in divide", category=RuntimeWarning)

# Run an Evidently drift report to compare the two datasets.
report = Report([DataDriftPreset()])
snapshot = report.run(reference_data=reference_data, current_data=current_data)

# In Evidently 0.7.x, the HTML export lives on the Snapshot returned by report.run(...).
snapshot.save_html("reports/mlops_evidently_report.html")

print("Saved report to reports/mlops_evidently_report.html")


Saved report to reports/mlops_evidently_report.html


## WhyLogs And WhyLabs

WhyLogs helps us summarize what model inputs and outputs look like over time.

You can think of it as creating a compact profile of a dataset or prediction batch. That profile can later help answer questions like:
- Are prediction scores changing a lot?
- Are some features missing more often than before?
- Do production inputs still look normal?

`whylabs-client` is also installed, so you can later send those profiles to WhyLabs if you want a hosted monitoring workflow.


In [6]:
# Import whylogs with a short alias used in most examples.
import whylogs as why
from whylogs.api.writer.local import LocalWriter

# Create a tiny scored dataset that looks like model prediction output.
scored_data = pd.DataFrame(
    {
        "prediction_score": [0.11, 0.31, 0.74, 0.82],
        "prediction_label": [0, 0, 1, 1],
        "monthly_charges": [49, 61, 79, 92],
    }
)

# Profile the batch so you can inspect feature distributions later.
profile = why.log(scored_data).profile()

# Use the explicit LocalWriter API to save the profile without deprecation warnings.
writer = LocalWriter(base_dir="artifacts", base_name="mlops_whylogs_profile.bin")
writer.write(profile)

print("Saved WhyLogs profile to artifacts/mlops_whylogs_profile.bin")


Saved WhyLogs profile to artifacts/mlops_whylogs_profile.bin


In [7]:
# Optional WhyLabs upload example
# from whylogs.api.writer.whylabs import WhyLabsWriter
#
# # Initialize the writer after setting WhyLabs credentials in your environment.
# writer = WhyLabsWriter()
#
# # Upload the profile view generated in the previous cell.
# writer.write(profile.view())


## DVC

DVC stands for Data Version Control.

It is similar to Git, but for large datasets and model files. It helps us answer questions like:
- Which dataset version was used for this experiment?
- If the data changes, can we track that change cleanly?

DVC is installed in the container as a CLI tool. The commands below are meant to be run in a terminal, not inside a notebook cell that changes repo state.


In [8]:
# These are the usual shell commands for putting a dataset under DVC control.
dvc_commands = [
    "dvc init",
    "dvc add notebooks/data/Telco-Customer-Churn.csv",
    "git add .",
    "git commit -m 'Track churn dataset with DVC'",
]

# Return the list so you can copy the commands into a terminal when ready.
dvc_commands


['dvc init',
 'dvc add notebooks/data/Telco-Customer-Churn.csv',
 'git add .',
 "git commit -m 'Track churn dataset with DVC'"]

## FastAPI Service

The local stack includes a starter FastAPI app.

FastAPI is a simple way to turn model logic into a web API. In practice, this is how another application might ask your model for predictions.

When you call services from inside this Jupyter notebook, use the Docker service name, not `localhost`.

Notebook-to-container URL:
- `http://fastapi:8000/health`

Browser URL on your machine:
- `http://localhost:8000/health`

Current endpoints:

- `GET /health`
- `GET /`
- `GET /metrics`


In [9]:
# Call the FastAPI service over the Docker network from inside the notebook container.
import requests

requests.get("http://fastapi:8000/health", timeout=5).json()


{'status': 'ok'}

## TensorFlow Serving And TorchServe

These are model-serving systems. Their job is to load trained models and answer prediction requests over HTTP.

In simple terms:
- TensorFlow Serving is commonly used for TensorFlow models
- TorchServe is commonly used for PyTorch models

These are provisioned as Docker services behind the `serving` profile.

From inside this notebook, use the Docker service names and internal container ports.

Notebook-to-container URLs:
- TensorFlow Serving model status: `http://tensorflow-serving:8501/v1/models/example_model`
- TensorFlow Serving predict: `http://tensorflow-serving:8501/v1/models/example_model:predict`
- TorchServe inference: `http://torchserve:8080`
- TorchServe management: `http://torchserve:8081`
- TorchServe metrics: `http://torchserve:8082`

Browser URLs on your machine:
- TensorFlow Serving model status: `http://localhost:8501/v1/models/example_model`
- TensorFlow Serving predict: `http://localhost:8501/v1/models/example_model:predict`
- TorchServe inference: `http://localhost:9181`
- TorchServe management: `http://localhost:9182`
- TorchServe metrics: `http://localhost:9183`

> Side note: the root URL `http://localhost:8501/` is not the main endpoint you usually use. TensorFlow Serving is loading a tiny demo model named `example_model`, and the useful endpoints are the model-status and predict routes above.

> Demo model behavior: `output = input * 2 + 1`
>
> This demo is intentionally simple. It is only here so you can learn how to call a served model and read the response.

> Example mapping:
> - `1.0 -> 3.0`
> - `2.5 -> 6.0`
> - `4.0 -> 9.0`

> Example curl request:
>
> ```bash
> curl -X POST http://localhost:8501/v1/models/example_model:predict \
>   -H "Content-Type: application/json" \
>   -d '{"instances":[1.0,2.5,4.0]}'
> ```

> Example JSON response:
>
> ```json
> {"predictions":[3.0,6.0,9.0]}
> ```


In [ ]:
# List both the Docker-network URLs used by notebook code and the browser URLs exposed on the host.
serving_endpoints = {
    "notebook": {
        "tensorflow_serving_status": "http://tensorflow-serving:8501/v1/models/example_model",
        "tensorflow_serving_predict": "http://tensorflow-serving:8501/v1/models/example_model:predict",
        "torchserve_inference": "http://torchserve:8080",
        "torchserve_management": "http://torchserve:8081",
        "torchserve_metrics": "http://torchserve:8082",
    },
    "browser": {
        "tensorflow_serving_status": "http://localhost:8501/v1/models/example_model",
        "tensorflow_serving_predict": "http://localhost:8501/v1/models/example_model:predict",
        "torchserve_inference": "http://localhost:9181",
        "torchserve_management": "http://localhost:9182",
        "torchserve_metrics": "http://localhost:9183",
    },
}

# Display the endpoint map for quick copy/paste use.
serving_endpoints


In [ ]:
# Send a demo request to the TensorFlow Serving example_model endpoint.
tf_demo_payload = {"instances": [1.0, 2.5, 4.0]}

# From inside the notebook container, call TensorFlow Serving over the Docker network.
tf_demo_response = requests.post(
    "http://tensorflow-serving:8501/v1/models/example_model:predict",
    json=tf_demo_payload,
    timeout=10,
)

# Show the JSON response so you can confirm the demo model is live.
tf_demo_response.json()


## Prometheus

Prometheus is a monitoring tool. It collects numbers over time, such as request counts, error counts, or response times.

This is useful because once a model API is live, we want to know whether it is healthy, fast, and stable.

Prometheus is available behind the `monitoring` profile and is configured to scrape the FastAPI starter metrics endpoint.

From inside this notebook use `http://prometheus:9090`, and from your browser use `http://localhost:9090`.


In [11]:
# Keep both notebook-internal and browser URLs handy.
prometheus_url = {
    "notebook": "http://prometheus:9090",
    "browser": "http://localhost:9090",
}

# Return the URL so you can open it directly from the notebook output.
prometheus_url


{'notebook': 'http://prometheus:9090', 'browser': 'http://localhost:9090'}

## Suggested Next Steps

If you want to continue from this starter notebook, these are the most natural next improvements:

1. Log the churn model from the marketing notebook into MLflow.
2. Register the best-performing version in MLflow Model Registry.
3. Add a real `/predict` endpoint to the FastAPI app.
4. Emit prediction logs to WhyLogs.
5. Generate a recurring Evidently drift report on scored batches.
6. Add Prometheus counters and latency histograms to the inference API.
